<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/01_Excerscise_Annotation_Execution_Spaces.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # I will be running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.02-Execution-Spaces/Sources/ach.h -nv -O Sources/ach.h

2026-09-03 20:23:47 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.02-Execution-Spaces/Sources/ach.h [2893/2893] -> "Sources/ach.h" [1]


In [6]:
%%writefile Sources/no-magic-execution-space-changes.cpp

#include "ach.h"
int main(){

  ach::where_am_I("CPU"); // main function always starts with CPU.

  thrust::universal_vector<int> vec{1};
  thrust::for_each(thrust::device, vec.begin(), vec.end(), [] __host__ __device__ (int) {ach::where_am_I("GPU");}); // execution policy on the GPU
  thrust::for_each(thrust::host, vec.begin(), vec.end(), [] __host__ __device__ (int) {ach::where_am_I("CPU");}); // execution policy on the CPU

  ach::where_am_I("CPU");
}

Overwriting Sources/no-magic-execution-space-changes.cpp


In [7]:
!nvcc -o /tmp/a.out --extended-lambda Sources/no-magic-execution-space-changes.cpp -x cu -arch=native
!/tmp/a.out

Correct! The function is invoked on CPU
Correct! The function is invoked on GPU
Correct! The function is invoked on CPU
Correct! The function is invoked on CPU


In [10]:
%%writefile Sources/port-thrust-to-gpu.cpp
#include "ach.h"

int main(){
  thrust::universal_vector<int> vec{1, 2, 3};
  thrust::for_each(thrust::device, vec.begin(), vec.end(),  [] __host__ __device__(int val){
    std::printf("printing %d on %s\n", val, ach::execution_space());
  });
}

Overwriting Sources/port-thrust-to-gpu.cpp


In [11]:
!nvcc -o /tmp/a.out --extended-lambda Sources/port-thrust-to-gpu.cpp -x cu -arch=native
!/tmp/a.out

printing 1 on GPU
printing 2 on GPU
printing 3 on GPU


- find median first on CPU

In [14]:
%%writefile Sources/port-sort-to-gpu.cpp
#include "ach.h"

float median(thrust::universal_vector<float> vec)
{
  std::sort(vec.begin(), vec.end());
  return vec[vec.size()/2];
}

int main(){
  float k = 0.5;
  float ambient_temp = 20;
  thrust::universal_vector<float> temp{42, 24, 50};
  auto transformation  = [=] __host__ __device__ (float temp) {return temp + k*(ambient_temp - temp);};

  std::printf("step median\n");
  for(int step = 0; step < 3; step++){
    thrust::transform(thrust::device, temp.begin(), temp.end(), temp.begin(), transformation);
    float median_temp = median(temp);
    std::printf("%d: %.2f\n", step, median_temp);
  }


}

Overwriting Sources/port-sort-to-gpu.cpp


In [15]:
!nvcc -o /tmp/a.out --extended-lambda Sources/port-sort-to-gpu.cpp -x cu -arch=native
!/tmp/a.out

step median
0: 31.00
1: 25.50
2: 22.75


Now on GPU

In [18]:
%%writefile Sources/port-sort-to-gpu.cpp
#include "ach.h"

float median(thrust::universal_vector<float> vec)
{
  thrust::sort(thrust::device, vec.begin(), vec.end());
  return vec[vec.size()/2];
}

int main(){
  float k = 0.5;
  float ambient_temp = 20;
  thrust::universal_vector<float> temp{42, 24, 50};
  auto transformation  = [=] __host__ __device__ (float temp) {return temp + k*(ambient_temp - temp);};

  std::printf("step median\n");
  for(int step = 0; step < 3; step++){
    thrust::transform(thrust::device, temp.begin(), temp.end(), temp.begin(), transformation);
    float median_temp = median(temp);
    std::printf("%d: %.2f\n", step, median_temp);
  }


}

Overwriting Sources/port-sort-to-gpu.cpp


In [19]:
!nvcc -o /tmp/a.out --extended-lambda Sources/port-sort-to-gpu.cpp -x cu -arch=native
!/tmp/a.out

step median
0: 31.00
1: 25.50
2: 22.75
